# Inspect the DFT test set and evaluate UMA-S
Run from the repository root. Install dependencies with `pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
import numpy as np
from ase.io import read
root = Path.cwd()
if not (root / 'data').exists(): root = root.parent
frames = read(root / 'data/dft_round0/test.extxyz', ':')
pred = read(root / 'data/predictions/uma-s-finetuned_test.extxyz', ':')
by_id = {a.info['calc_id']: a for a in pred}
a = frames[0]
print('Number of test configurations:', len(frames))
print('Example:', a.info['calc_id'], 'energy (eV):', a.info['REF_energy'])
print('Force array shape:', a.arrays['REF_forces'].shape)


In [ ]:
errors = []
for a in frames:
    b = by_id[a.info['calc_id']]
    assert np.allclose(a.arrays['REF_forces'], b.arrays['REF_forces'], atol=1e-7)
    errors.extend((b.arrays['MLIP_forces'] - a.arrays['REF_forces']).ravel())
print('All-atom force component RMSE (eV/Å):', np.sqrt(np.mean(np.square(errors))))
